In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q "/content/drive/MyDrive/archive.zip" -d "/content/dataset"
!pip install ultralytics
from ultralytics import YOLO
model = YOLO("yolo11n.pt")

In [ ]:
import os, cv2, yaml, random, numpy as np
from PIL import Image
from glob import glob
from matplotlib import pyplot as plt
from torchvision import transforms as T

class Visualization:

    def __init__(self, root, data_types, n_ims, rows, cmap = None):

        self.n_ims, self.rows = n_ims, rows
        self.cmap, self.data_types  = cmap, data_types
        self.colors = ["firebrick", "darkorange", "blueviolet"]

        self.get_cls_names(); self.get_bboxes()

    def get_cls_names(self):

        with open(f"{root}/data.yaml", 'r') as file: data = yaml.safe_load(file)
        # Extract class names
        class_names = data['names']

        # Create a dictionary with index as key and class name as value
        self.class_dict = {index: name for index, name in enumerate(class_names)}

    def get_bboxes(self):

        self.vis_datas, self.analysis_datas, self.im_paths = {}, {}, {}
        for data_type in self.data_types:
            all_bboxes, all_analysis_datas = [], {}
            im_paths = glob(f"{root}/{data_type}/images/*")

            for idx, im_path in enumerate(im_paths):
                # if idx == 3: break
                bboxes = []
                if ".png" in im_path: lbl_path = im_path.replace(".png", ".txt")
                elif ".jpg" in im_path: lbl_path = im_path.replace(".jpg", ".txt")
                lbl_path  = lbl_path.replace(f"{data_type}/images/", f"{data_type}/labels/")
                if not os.path.isfile(lbl_path): continue
                meta_data = open(lbl_path).readlines()
                for data in meta_data:
                    # Split the string by space and strip the newline character
                    # Change here
                    parts = data.strip().split()[:5]
                    cls_name = self.class_dict[int(parts[0])]
                    # Convert first element to integer and the rest to float
                    bboxes.append([cls_name] + [float(x) for x in parts[1:]])
                    if cls_name not in all_analysis_datas: all_analysis_datas[cls_name] = 1
                    else: all_analysis_datas[cls_name] += 1
                all_bboxes.append(bboxes)

            self.vis_datas[data_type] = all_bboxes; self.analysis_datas[data_type] = all_analysis_datas; self.im_paths[data_type] = im_paths

    def plot(self, rows, cols, count, im_path, bboxes):

        plt.subplot(rows, cols, count)
        or_im = np.array(Image.open(im_path).convert("RGB"))
        height, width, _ = or_im.shape

        for bbox in bboxes:

            class_id, x_center, y_center, w, h = bbox

            # Convert YOLO format to pixel values
            x_min = int((x_center - w / 2) * width)  # x_min
            y_min = int((y_center - h / 2) * height)  # y_min
            x_max = int((x_center + w / 2) * width)  # x_max
            y_max = int((y_center + h / 2) * height)  # y_max

            color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
            cv2.rectangle(img = or_im, pt1 = (x_min, y_min), pt2 = (x_max, y_max), color = color, thickness = 3)
        plt.imshow(or_im)
        plt.axis("off"); plt.title(f"There is (are) {len(bboxes)} object(s) in the image.")

        return count + 1

    def vis(self, save_name):

        print(f"{save_name.upper()} Data Visualization is in process...\n")
        assert self.cmap in ["rgb", "gray"], "Please choose rgb or gray cmap"
        if self.cmap == "rgb": cmap = "viridis"
        cols = self.n_ims // self.rows; count = 1

        plt.figure(figsize = (25, 20))

        indices = [random.randint(a = 0, b = len(self.vis_datas[save_name]) - 1) for _ in range(self.n_ims)]

        for idx, index in enumerate(indices):

            if count == self.n_ims + 1: break

            im_path, bboxes = self.im_paths[save_name][index], self.vis_datas[save_name][index]

            count = self.plot(self.rows, cols, count, im_path = im_path, bboxes = bboxes)

        plt.show()

    def data_analysis(self, save_name, color):

        print("Data analysis is in process...\n")

        width, text_width, text_height = 0.7, 0.05, 2
        cls_names = list(self.analysis_datas[save_name].keys()); counts = list(self.analysis_datas[save_name].values())

        _, ax = plt.subplots(figsize = (30, 10))
        indices = np.arange(len(counts))

        ax.bar(indices, counts, width, color = color)
        ax.set_xlabel("Class Names", color = "black")
        ax.set_xticklabels(cls_names)
        ax.set(xticks = indices, xticklabels = cls_names)
        ax.set_ylabel("Data Counts", color = "black")
        ax.set_title(f"{save_name.upper()} Dataset Class Imbalance Analysis")

        for i, v in enumerate(counts): ax.text(i - text_width, v + text_height, str(v), color = "royalblue")

    def visualization(self): [self.vis(save_name) for save_name in self.data_types]

    def analysis(self): [self.data_analysis(save_name, color) for (save_name, color) in zip(self.data_types, self.colors)]

root = "/content/dataset"
vis = Visualization(root = root, data_types = ["train", "valid", "test"], n_ims = 20, rows = 5, cmap = "rgb")
vis.analysis()

In [ ]:
train_results = model.train(
    data=f"{root}/data.yaml",  # path to dataset YAML
    epochs=10,                 # number of training epochs
    imgsz=480,                 # training image size
    device=0                   # <-- Changed to 0 for a single GPU
)
Image.open(f"runs/detect/train/results.png")

In [ ]:
inference_results = model(f"{root}/test/images", device = 0, verbose = False)
def inference_vis(res, n_ims, rows):
    cols = n_ims // rows
    plt.figure(figsize = (20, 10))
    for idx, r in enumerate(res):
        if idx == n_ims: break
        plt.subplot(rows, cols, idx + 1)
        or_im_rgb = np.array(Image.open(r.path).convert("RGB"))
        if idx == n_ims: break
        for i in r:
            for bbox in i.boxes:
                box = bbox.xyxy[0]
                x1, y1, x2, y2 = box
                coord1, coord2 = (int(x1), int(y1)), (int(x2), int(y2))
                cv2.rectangle(or_im_rgb, coord1, coord2, color=(255,0,0), thickness=2)
        plt.imshow(or_im_rgb)
        plt.title(f"Image#{idx + 1}")
        plt.axis("off")
inference_vis(res = inference_results, n_ims = 15, rows = 3)

In [ ]:
import os

# This will print out every single file inside your train folder
saved_files = os.listdir('runs/detect/train')
for file in saved_files:
    print(file)

In [ ]:
from IPython.display import Image, display

print("--- Overall Training Results ---")
# This shows the graphs of how well the AI learned
display(Image(filename='runs/detect/train/results.png', width=800))

print("\n--- AI Predictions on Validation Images ---")
# This shows actual pictures of fish with the AI's bounding boxes!
display(Image(filename='runs/detect/train/val_batch0_pred.jpg', width=800))

In [ ]:
import os
import glob
import random
from IPython.display import Image, display

# 1. Grab a random image from your test dataset
test_images = glob.glob('/content/dataset/test/images/*.*')
my_test_image = random.choice(test_images)
print(f"🐟 Testing model on: {my_test_image}")

# 2. Run the YOLO prediction using your custom trained weights
# (Note: make sure your best.pt is in the runs/detect/train/weights/ folder)
!yolo predict model=runs/detect/train/weights/best.pt source="{my_test_image}" conf=0.25

# 3. Find where YOLO saved the resulting image
# YOLO creates a new 'predict' folder for every test (predict, predict2, predict3, etc.)
predict_folders = glob.glob('runs/detect/predict*')
latest_predict_folder = max(predict_folders, key=os.path.getmtime)

# Get the filename of the image we just tested
image_name = os.path.basename(my_test_image)
result_path = f"{latest_predict_folder}/{image_name}"

# 4. Display the final image with bounding boxes!
print(f"\n✅ Prediction saved to: {result_path}")
display(Image(filename=result_path, width=800))

In [ ]:
import os
import glob
from IPython.display import Image, display

# 1. Paste your internet image link inside the quotes!
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/ad/Amphiprion_ocellaris_%28Clown_anemonefish%29_by_Nick_Hobgood.jpg/1280px-Amphiprion_ocellaris_%28Clown_anemonefish%29_by_Nick_Hobgood.jpg"

# Download the image to Colab and save it as 'internet_fish.jpg'
!wget -q -O internet_fish.jpg "{image_url}"

print("🐟 Testing model on internet image...")

# 2. Run the YOLO prediction on the downloaded image
!yolo predict model=runs/detect/train/weights/best.pt source="internet_fish.jpg" conf=0.25

# 3. Find where YOLO saved the resulting image
predict_folders = glob.glob('runs/detect/predict*')
latest_predict_folder = max(predict_folders, key=os.path.getmtime)
result_path = f"{latest_predict_folder}/internet_fish.jpg"

# 4. Display the final image with bounding boxes!
print(f"\n✅ Prediction saved to: {result_path}")
display(Image(filename=result_path, width=800))